In [1]:
# Author: Gergely Zahoranszky-Kohalmi, PhD
#
# Email: gergely.zahoranszky-kohalmi@nih.gov
#
# Organization: National Center for Advancing Translational Sciences
#


In [2]:
# environment: routesim

import pandas as pd

import os

import requests
import json

In [3]:

FNAME_IN_ALL = []

DIR_IN = '../data/output/predicted_routes/rxns/'
DIR_OUT = '../data/output/predicted_routes/mapped_rxns/'

URL_RXNMAPPER = 'http://localhost:8002/syngps-app/api/v1/reaction_utils/atommap'



In [4]:
# Functions

def rxsmiles2mappedrxsmiles (rxsmiles):
    """
        Search depths: maximal number of reactions steps to explore.
    """
    
    # Base URL for your API

    

    
    try:
        # Step 1: Make POST request to /api/test
        print("Atommapping reaction ...")
        
        # Optional: Include data in the first request if needed




        rxnmapper_payload = {
            "smiles": rxsmiles
        }



        # synth_route_search_payload = {
        #     "target_molecule_inchikey": target_molecule_inchikey,
        #     "reaction_steps": search_depth,
        #     "query_type": search_type,
        #     "leaves_as_sm": leaves_as_sm,
        #     "include_availability_info": False,
        #     "annotate_reactions": False,
        #     "graph_backend": "memgraph",
        #     "top_n_routes": top_n,
        #     "include_route_candidates": False,
        #     "include_combination_graphs": False
        # }
        
        response1 = requests.post(
            url = URL_RXNMAPPER,
            json = rxnmapper_payload,
            headers={'Content-Type': 'application/json',
                     'accept': 'application/json'})
        
        # Check if the request was successful
        #response1.raise_for_status()
        
        # Get JSON data from response
        received_data = response1.json()
        #print(f"Received data: {json.dumps(received_data, indent=2)}")


        return (received_data['mapped_rxsmiles'])
    
    except requests.exceptions.RequestException as e:
    
        print(f"Error occurred: {e}")

        return (None)


def annotate_routes_by_atommapping (fname_in, fname_out):
    df = pd.read_csv (fname_in, sep = '\t')

    print (df)

    df['real_mapped_rxsmiles'] = df.apply(lambda x: rxsmiles2mappedrxsmiles (x['rxsmiles']), axis = 1)

    df = df[['tm_inchikey', 'rxsmiles', 'route_index', 'real_mapped_rxsmiles']].copy()

    df = df.rename (columns = {
        'real_mapped_rxsmiles': 'mapped_reaction_smiles'
    })

    df.to_csv(fname_out, sep = '\t', index = False)






In [5]:
# Ensure DIR_OUT exists
if not os.path.exists(DIR_OUT):
    os.makedirs(DIR_OUT)

search_obj = os.scandir(DIR_IN)

for dir_item in search_obj:
    if dir_item.is_file():
        
        FNAME_IN_ALL.append(DIR_IN + dir_item.name)

print(FNAME_IN_ALL)

idx = 1

for fname in FNAME_IN_ALL:
    fname_out = fname.split('/')[-1]                    \
                                    .strip()            \
                                    .split('.')[0]      \
                                    .strip()          \
                                    + '_predicted_mapped_rxn.tsv'
    
    fname_out = DIR_OUT + fname_out
    
    print (f'[*] Processing route nr. {idx} of {len(FNAME_IN_ALL)} ..')
    
    idx += 1

    try:
        annotate_routes_by_atommapping (fname, fname_out)
    
    except:
        print (f'[W] Some of the reactions could not be atommapped in input file: {fname} .')

    print (f'[*] .. done.')



['../data/output/predicted_routes/rxns/ZVERWTXKKWSSHH-UHFFFAOYSA-N_predicted_route_predicted_rxn.tsv', '../data/output/predicted_routes/rxns/AXZKSUNGRDJIFL-UHFFFAOYSA-N_predicted_route_predicted_rxn.tsv', '../data/output/predicted_routes/rxns/UTRFILHTKFEHEN-UHFFFAOYSA-N_predicted_route_predicted_rxn.tsv', '../data/output/predicted_routes/rxns/POSWICCRDBKBMH-UHFFFAOYSA-N_predicted_route_predicted_rxn.tsv', '../data/output/predicted_routes/rxns/OPYBJNYVSLHSQI-UHFFFAOYSA-N_predicted_route_predicted_rxn.tsv', '../data/output/predicted_routes/rxns/GNQGSUOEYQIORN-UHFFFAOYSA-N_predicted_route_predicted_rxn.tsv', '../data/output/predicted_routes/rxns/BYXNPBBWWSOSDA-VWLOTQADSA-N_predicted_route_predicted_rxn.tsv', '../data/output/predicted_routes/rxns/PJKXXTQGJSHEOQ-UHFFFAOYSA-N_predicted_route_predicted_rxn.tsv', '../data/output/predicted_routes/rxns/ONGPJYSSZQHFBU-UHFFFAOYSA-N_predicted_route_predicted_rxn.tsv', '../data/output/predicted_routes/rxns/WUFUURSWOJROKY-UHFFFAOYSA-N_predicted_route

In [6]:
print ('[Done.]')

[Done.]


In [7]:
# References:

# Ref: https://github.com/rxn4chemistry/rxnmapper
#
